In [0]:
%sql
-- Ensure the final RAG table exists and has the required columns
SELECT
  chunk_id,
  text,
  embedding
FROM tdis_dev_data_catalog.tdir.optimized_rag_chunks
LIMIT 10;


# Download Embedding model

In [0]:
# Download HuggingFace model snapshot into UC Volume
from huggingface_hub import snapshot_download
import os

HF_MODEL_ID = "DMIR01/DMRetriever-33M"
MODEL_DIR = "/Volumes/tdis_dev_data_catalog/tdir/tdir/models/DMRetriever-33M"

os.makedirs(MODEL_DIR, exist_ok=True)

snapshot_download(
    repo_id=HF_MODEL_ID,
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False
)

print("Downloaded to:", MODEL_DIR)
display(dbutils.fs.ls(MODEL_DIR))


In [0]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

MODEL_DIR = "/Volumes/tdis_dev_data_catalog/tdir/tdir/models/DMRetriever-33M"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModel.from_pretrained(MODEL_DIR)
model.eval()


In [0]:
def encode_query(text: str):
    """
    Encode a single query using DMRetriever-33M.
    Output: list[float], length = 384
    """
    with torch.no_grad():
        inputs = tokenizer(
            text,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # [1, seq_len, hidden]

        mask = inputs["attention_mask"].unsqueeze(-1)
        pooled = (last_hidden * mask).sum(dim=1) / mask.sum(dim=1)

        pooled = F.normalize(pooled, p=2, dim=1)

    return pooled[0].cpu().tolist()


In [0]:
qv = encode_query("Harris County flood damage estimation methodology")
print(len(qv), qv[:5])


# Run a query

In [0]:
from dbx_retrieve import embed_query, register_query_vector, vector_search
import warnings

warnings.filterwarnings('ignore')

INDEX_FQN = "tdis_dev_data_catalog.tdir.rag_chunks_vs"
TOP_K = 10

query = "Harris County flood damage estimation methodology"

# 1) Embed query
qv = embed_query(query)

# 2) Register vector to Spark
register_query_vector(spark, qv)

# 3) Vector search
df = vector_search(spark, INDEX_FQN, top_k=TOP_K)

display(df)


In [0]:
from dbx_retrieve import embed_query, register_query_vector, vector_search
import warnings

warnings.filterwarnings('ignore')

INDEX_FQN = "tdis_dev_data_catalog.tdir.optimized_rag_chunks_vs"
TOP_K = 10

query = "Harris County flood damage estimation methodology"

# 1) Embed query
qv = embed_query(query)

# 2) Register vector to Spark
register_query_vector(spark, qv)

# 3) Vector search
df = vector_search(spark, INDEX_FQN, top_k=TOP_K)

display(df)
